# Aşama 8: TXS 0506+056'nın patlaması

10 yılın tamamına bakınca TXS 0506+056 zayıf görünüyordu (~1.4σ). Sebebi: bu kaynağın nötrinoları sürekli değil, **birkaç aylık bir patlamada** gelmiş. 10 yılın arka planı o kısa sinyali boğuyor.

Çözüm: Likelihood'a **zaman** boyutunu ekleyip "hangi zaman aralığında fazlalık var?" diye sormak. Patlamanın merkezini (T₀) ve süresini (σ_T) bir ızgarada tarıyoruz. Doğru pencereye bakınca arka plan dramatik biçimde azalıyor, sinyal ortaya çıkıyor.

In [ ]:
import sys; sys.path.append('../src')
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import icecube_ps as ps
importlib.reload(ps)
assert float(ps.SURUM) >= float('8.0'), f"src/icecube_ps.py eski sürüm ({getattr(ps, 'SURUM', 'yok')}). Yenisini kopyala, kernel'i yeniden başlat."
try:
    from tqdm.auto import tqdm          # ilerleme çubuğu
except ImportError:
    tqdm = None

pd.set_option('display.float_format', '{:.3g}'.format)
DATA, RESULTS = Path('../data'), Path('../results')
ev = ps.veri_yukle(DATA)
B = ps.ArkaPlan(ev['dec'].values)
print('icecube_ps', ps.SURUM, '| olay:', f'{len(ev):,}')

def yil(mjd):
    return 2000 + (np.asarray(mjd) - 51544.5) / 365.25

def tarih(mjd):
    return pd.to_datetime(mjd - 40587, unit='D').strftime('%d %b %Y')

T_canli = sum(ps.canli_sureler_yukle(DATA).values()) / 86400
print(f'Toplam canlı süre: {T_canli:.0f} gün')

## 1. Olaylar zamanda nasıl dağılmış?

TXS'nin 1°'lik çevresindeki olaylar, zamana göre. Dikey eksen kanıt gücü (Aşama 3'teki S/B). Bir patlama varsa, belli bir dönemde yukarıda kümelenmiş noktalar görmeliyiz.

In [ ]:
ad = 'TXS 0506+056'
Z = ps.ZamanAnalizi(ev, B, *ps.KAYNAKLAR[ad], T_canli)
sb_u, t_u = Z.uzay_sb()
psi = ps.acisal_uzaklik(Z.ra, Z.dec, *ps.KAYNAKLAR[ad])
yakin = psi < 1

fig, ax = plt.subplots(figsize=(12, 4))
ax.scatter(yil(Z.t[yakin]), np.log10(ps.sinyal_pdf(psi[yakin], Z.sig[yakin]) / Z.B[yakin]),
           s=10, color='#2e4057', alpha=0.6)
ax.set_xlabel('yıl'); ax.set_ylabel('log₁₀(S/B)')
ax.set_title(f'{ad}: 1° içindeki olaylar zamanda'); ax.grid(alpha=0.3)
plt.savefig(RESULTS / 'asama8_zaman_olaylar.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Patlama araması

Her (T₀, σ_T) çifti için fit. σ_T 5 günden 1500 güne, T₀ 10 yıl boyunca 20 gün aralıkla. Isı haritası hangi zaman penceresinin en çok sinyal verdiğini gösteriyor.

In [ ]:
en_iyi, NS, TS = Z.en_iyi()
print(f"En iyi patlama: merkez {tarih(en_iyi['T0'])}, süre σ_T = {en_iyi['sigma_T']:.0f} gün")
print(f"nₛ = {en_iyi['ns']:.1f} nötrino,  TS = {en_iyi['TS']:.1f}")

fig, ax = plt.subplots(figsize=(12, 4.5))
m = ax.pcolormesh(yil(Z.t0_izgara), np.arange(len(Z.sigma_izgara)), np.clip(TS, 0, None),
                  cmap='magma', shading='nearest')
ax.set_yticks(range(len(Z.sigma_izgara)))
ax.set_yticklabels([f'{s:.0f}' for s in Z.sigma_izgara])
ax.plot(yil(en_iyi['T0']), list(Z.sigma_izgara).index(en_iyi['sigma_T']), 'x', color='#66ff99', ms=14, mew=2.5)
fig.colorbar(m, label='TS')
ax.set_xlabel('patlama merkezi T₀ (yıl)'); ax.set_ylabel('patlama süresi σ_T (gün)')
ax.set_title(f'{ad}: zaman penceresi taraması')
plt.savefig(RESULTS / 'asama8_zaman_isi_haritasi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Anlamlılık

Bu sefer Wilks kullanamıyoruz (iki ek parametre taradık, durum karmaşık), o yüzden tamamen Monte Carlo: RA'ları karıştırıp aynı zaman taramasını tekrar ediyoruz. Zaman ızgarasını taramak da "binlerce zaman penceresine bakmak" demek; bu karıştırma o etkiyi de otomatik düzeltiyor.

⏱️ 2000 deneme. Parça parça kaydediliyor, durdurup devam edebilirsin.

In [ ]:
N_DENEME = 2000
dosya = RESULTS / 'asama8_ts_arka.npy'
ts_arka = list(np.load(dosya)) if dosya.exists() else []
rng = np.random.default_rng(800 + len(ts_arka))
dongu = range(len(ts_arka), N_DENEME)
for i in (tqdm(dongu, desc='karıştırma') if tqdm else dongu):
    ts_arka.append(Z.en_iyi(rng.uniform(0, 360, len(Z.ra)))[0]['TS'])
    if i % 100 == 99 or i == N_DENEME - 1:
        np.save(dosya, np.array(ts_arka))
ts_arka = np.array(ts_arka)

p_zaman = (np.sum(ts_arka >= en_iyi['TS']) + 1) / (len(ts_arka) + 1)
asama3 = pd.read_csv(RESULTS / 'asama3_likelihood.csv').set_index('kaynak')
print(f"Zamandan bağımsız (Aşama 3):  {asama3.loc[ad, 'sigma_Wilks']:.2f}σ")
print(f"Patlama araması (Aşama 8):    {ps.sigma(p_zaman):.2f}σ   (p = {p_zaman:.2g})")
if np.sum(ts_arka >= en_iyi['TS']) == 0:
    print(f"  Not: hiçbir deneme geçemedi, gerçek değer ≥ bu ({len(ts_arka)} deneme sınırı)")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={'width_ratios': [1.3, 1]})

ax = axs[0]
t_ek = np.linspace(Z.t0_izgara.min(), Z.t0_izgara.max(), 3000)
g = np.exp(-0.5 * ((t_ek - en_iyi['T0']) / en_iyi['sigma_T']) ** 2)
ax.fill_between(yil(t_ek), 0, g * 4.5, color='#d1495b', alpha=0.15, label='en iyi patlama penceresi')
ax.scatter(yil(Z.t[yakin]), np.log10(ps.sinyal_pdf(psi[yakin], Z.sig[yakin]) / Z.B[yakin]),
           s=10, color='#2e4057', alpha=0.6)
ax.set_xlabel('yıl'); ax.set_ylabel('log₁₀(S/B)'); ax.legend(loc='upper left')
ax.set_title('Olaylar ve bulunan patlama')

ax = axs[1]
ax.hist(ts_arka, bins=50, color='#7a8fb8', label='karıştırılmış gökyüzleri')
ax.axvline(en_iyi['TS'], color='#d1495b', lw=2.5, label=f"gözlenen TS = {en_iyi['TS']:.1f}")
ax.set_yscale('log'); ax.set_xlabel('TS (en iyi patlama)'); ax.legend()
ax.set_title(f'Patlama araması: {ps.sigma(p_zaman):.1f}σ')
plt.tight_layout()
plt.savefig(RESULTS / 'asama8_patlama.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Kontrol: Diğer kaynaklar

Aynı patlama aramasını diğer üç kaynakta da yapıyoruz (sadece en iyi pencere, anlamlılık yok). Sürekli parlayan bir kaynakta en iyi pencerenin genelde **uzun** (σ_T büyük) çıkmasını bekleriz. Kısa bir pencere seçip TS'si düşük kalan kaynaklar ise büyük ihtimalle sadece arka plan dalgalanmasına takılmıştır. TXS ile diğerlerini bu gözle karşılaştır.

In [ ]:
satirlar = []
for k, (rk, dk) in ps.KAYNAKLAR.items():
    b = ps.ZamanAnalizi(ev, B, rk, dk, T_canli).en_iyi()[0]
    satirlar.append(dict(kaynak=k, merkez=tarih(b['T0']), sure_gun=b['sigma_T'], ns=b['ns'], TS=b['TS']))
tablo8 = pd.DataFrame(satirlar)
tablo8.to_csv(RESULTS / 'asama8_kaynaklar.csv', index=False)
tablo8

## Yorumlarken

- **Önce/sonra:** Aynı kaynak, aynı veri. 10 yıla birden bakınca zayıf, doğru zaman penceresine bakınca güçlü. Nötrino astronomisinde kaynakların bir kısmının "yanıp sönen" (değişken) olduğunun doğrudan gösterimi.
- **IceCube ile karşılaştırma:** IceCube 2018'de TXS 0506+056'nın 2014–15 patlamasını duyurdu. Ancak bizim kullandığımız 2021 veri sürümünde 2012–2015 olayları yeniden işlendi ve IceCube'un kendi raporuna göre bu, TXS'nin anlamlılığını değiştirdi. Bu yüzden bizim sayımız 2018 makalesindekinden farklı (muhtemelen düşük) çıkabilir; bu beklenen bir şey.
- **Sadeleştirme:** Zaman sinyalini tek bir Gauss patlama olarak modelledik ve dedektörün kapalı olduğu kısa aralıkları yok saydık.